In [24]:
#import relevant libraries
import os
#from scipy import stats

import numpy as np
#import scipy as sp
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm
import seaborn as sns
import dabest
import NLCLIMB  # Using the new multistate version
import itertools
from datetime import datetime
date = datetime.today().strftime('%Y%m%d')
from statistics import mean
from textwrap import wrap


import dabest
import plotly.express as px 
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from plotly.graph_objects import Layout
#from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import NLMATH
#NOTE: SUPPRESSES WARNINGS!

import warnings


warnings.simplefilter(action="ignore", category=RuntimeWarning)
warnings.simplefilter(action="ignore", category=UserWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

In [25]:
#initial file processing
workcomp = "C:\\Users\\User"
computer2 = "C:\\Users\\lnico"
officecomp = "C:\\Users\\Star"
homecomp = "D:"
titledpath = homecomp

#Comment depending on project type
#eopn3work = "N"
eopn3work = "Y"
eopn3work = "tgt"

if eopn3work == "Y":
    whomst ="NL"
    filedir = "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\eOPN3 manuscript\\Data compilation\\2. Processed\\" + whomst + "\\"
    savedir = titledpath + "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\eOPN3 manuscript\\Data compilation\\3. Compiled\\"  + whomst + "\\"
    
if eopn3work == "tgt":
    filedir = "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\eOPN3 manuscript\\Data compilation\\Together\\2. Processed\\" 
    savedir = titledpath + "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\eOPN3 manuscript\\Data compilation\\Together\\3. Compiled\\" 
    
else:
    filedir = "\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\Data Compilation\\Falling_New\\"
    savedir = titledpath + filedir + "Compilation with delta\\2025meandiffcollection\\"
    
openPath = titledpath + filedir
files = os.listdir(openPath)

#identifying genotypes
responder = "eOPN3"
respondercsv = responder + ".csv"
wt = "w1118"

In [26]:
lstnew=[]

#lstnew should be the list of names you want to process the files with. Only choose one

#if you want to process all the names in the filedir

# for file_no in os.listdir(openPath): 
#     if respondercsv in file_no and "w1118" not in file_no :   
#         f = os.path.join(openPath, file_no)
#         dfe=pd.read_csv(f)
#         exptdf = dfe.drop(dfe.columns[[0]],axis = 1)
#         driver = file_no.split(" ")[0]
#         lstnew.append(driver)
# lst = lstnew.copy()

#processing ONLY specific names

if "eOPN3" in responder:
    lst = ["46416jus", "elav", "nSyb", "OK371", "vGlut"]
    lst= ['elav']

if responder == "ACR [ATR]":
    lst = ["elav", "46416jus", "vGlut"]
    
if responder == "ACR":
    lst = ['OK371']
    

print(lst)

['elav']


## With WT samples

In [27]:
# Process data with all three state comparisons
all_comparisons = pd.DataFrame()
comparison_types = ['DARK-FULL', 'DARK-RECOVERY']

for n in lst:
    driver = n
    print(f"Processing {n}...")
    transgenic = driver + " x " + responder
    filename = openPath + transgenic + ".csv"
    filenamewt = openPath + wt+"_"+ transgenic + ".csv"

    dfe=pd.read_csv(filename)
    dfw= pd.read_csv(filenamewt)

    exptdf = dfe.drop(dfe.columns[[0]],axis = 1)
    wtdf = dfw.drop(dfw.columns[[0]],axis = 1)

    dfexpt = NLCLIMB.fivesecondrule(NLCLIMB.generation(exptdf, driver))
    dfwt = NLCLIMB.fivesecondrule(NLCLIMB.generation(wtdf, wt))
    
    # Calculate metrics for all states (including Recovery)
    df_f = NLMATH.fallingocc(dfexpt, dfwt).reset_index(drop=True) 
    df_sp = NLMATH.ospeed(dfwt, dfexpt).reset_index(drop=True)
    df_bsp = NLMATH.bspeed(NLMATH.boutspeed(dfexpt), NLMATH.boutspeed(dfwt)).reset_index(drop=True)
    df_h = NLMATH.totalheight(dfexpt, dfwt).reset_index(drop=True)
    df_maxv = pd.concat([NLMATH.maxvelocity(dfexpt, "Expt"), NLMATH.maxvelocity(dfwt, "WT")], axis = 0).reset_index(drop=False)
    
    # Process each comparison type
    for comparison in comparison_types:
        print(f"  Computing {comparison} comparison...")
        
        dfs2 = NLMATH.deltaversion_multistate(df_sp, "Velocity", "speed", comparison)
        dfh2 = NLMATH.deltaversion_multistate(df_h, "Y", "height", comparison)
        dff2_prop = NLMATH.deltaversion_binary_multistate(df_f, "binary_fallvalue", "fallprop", comparison)
        
        # Combine all metrics for this comparison
        dftotal = pd.concat([dfs2, dfh2, dff2_prop], axis = 1)
        dftotal['MBON'] = n
        dftotal['State_Comparison'] = comparison
        dftotal['comparison_type'] = comparison  # Keep for consistency
        
        #samplesize
        samplesizelst = []
        for j in dfexpt.columns.unique().tolist()[2:]:
            number = j.split("_")[-1]
            samplesizelst.append(int(number))
        samplesize = len(list(set(samplesizelst)))
        dftotal['Sample_size'] = samplesize
        
        # Save individual comparison file
        dftotal.set_index("MBON", inplace = True)
        comparison_suffix = comparison.replace('-', '_')
        dftotal.to_csv(savedir + n + " x " + responder + f" allstats_{comparison_suffix}" + "_wtcomparison.csv")
        dftotal.reset_index(inplace=True)
    
print("\nProcessing complete!")

Processing elav...
  Computing DARK-FULL comparison...
  Computing DARK-RECOVERY comparison...

Processing complete!


## If want to compare to without WT samples

In [28]:
# Process data with all three state comparisons
all_comparisons = pd.DataFrame()
comparison_types = ['DARK-FULL', 'DARK-RECOVERY']

for n in lst:
    driver = n
    print(f"Processing {n}...")
    transgenic = driver + " x " + responder
    filename = openPath + transgenic + ".csv"
    filenamewt = openPath + wt+"_"+ transgenic + ".csv"

    dfe=pd.read_csv(filename)
    dfw= pd.read_csv(filenamewt)

    exptdf = dfe.drop(dfe.columns[[0]],axis = 1)
    wtdf = dfw.drop(dfw.columns[[0]],axis = 1)

    dfexpt = NLCLIMB.fivesecondrule(NLCLIMB.generation(exptdf, driver))
    dfwt = NLCLIMB.fivesecondrule(NLCLIMB.generation(wtdf, wt))
    
    # Calculate metrics for all states (including Recovery)
    df_sp = NLMATH.ospeed(dfwt, dfexpt).reset_index(drop=True)
    df_h = NLMATH.totalheight(dfexpt, dfwt).reset_index(drop=True)
    
    # Process each comparison type
    for comparison in comparison_types:
        print(f"  Computing {comparison} comparison...")
        
        dfs2 = NLMATH.deltaversion_baseline_multistate(df_sp, "Velocity", "speed", comparison)
        dfh2 = NLMATH.deltaversion_baseline_multistate(df_h, "Y", "height", comparison)
        dff2_prop = NLMATH.deltaversion_binarybaseline_multistate(df_f, "binary_fallvalue", "fallprop", comparison)
        
        # Combine all metrics for this comparison
        dftotal = pd.concat([dfs2, dfh2, dff2_prop], axis = 1)
        dftotal['MBON'] = n
        dftotal['State_Comparison'] = comparison
        dftotal['comparison_type'] = comparison  # Keep for consistency
        
        #samplesize
        samplesizelst = []
        for j in dfexpt.columns.unique().tolist()[2:]:
            number = j.split("_")[-1]
            samplesizelst.append(int(number))
        samplesize = len(list(set(samplesizelst)))
        dftotal['Sample_size'] = samplesize
               
        # Save individual comparison file
        dftotal.set_index("MBON", inplace = True)
        comparison_suffix = comparison.replace('-', '_')
        dftotal.to_csv(savedir + n + " x " + responder + f" allstats_{comparison_suffix}.csv")
        dftotal.reset_index(inplace=True)
    
print("\nProcessing complete!")

Processing elav...
  Computing DARK-FULL comparison...
  Computing DARK-RECOVERY comparison...

Processing complete!
